# 🤖 Notebook 3 — Deploy Specialist Agents

This notebook builds and deploys the five specialist agents that power the Product Finder.  
The orchestrator is deployed separately in Notebook 5 **after** these agents are active.

## Agents deployed here
| Agent | Role | Persona |
|-------|------|---------|
| `pf-contextualizer` | Extract intent, entities, risk tier | All |
| `pf-product-intelligence` | Search product catalog and recommend | All |
| `pf-compatibility` | Check product-pair compatibility + confidence | All |
| `pf-aligner` | Validate final answer matches intent | All |
| `pf-sample-request` | Process sample requests (authenticated) | external_customer only |

## Pattern
Each specialist agent:
- Is a Foundry **hosted agent** (container running the Responses API protocol)
- Routes all LLM calls through APIM via the BYO Gateway connection created by the Notebook 1 model-access contract
- Loads product and compatibility context from Azure AI Search, which is the retrieval source of truth for this use case
- Gets **Foundry User** RBAC on the Foundry account after deployment; no direct APIM subscription is assigned to each hosted agent identity
- Uses Azure AI Search through the workshop API key configuration in this scenario, so Search RBAC is not required for the agent identity
- Returns **structured JSON** that the orchestrator bundles

## Prerequisites
- Notebook 1, 2 completed successfully
- Notebook 1 created the Product Finder model-access contract and the BYO Gateway connection `Product-Finder-DEV-LLM` in your Foundry project  

In [ ]:
import sys, json, pathlib, subprocess, re

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing shared/utils.py and workshop/product-finder.")

repo_root = find_repo_root(pathlib.Path.cwd())
shared_dir = repo_root / "shared"
sys.path.insert(0, str(shared_dir))
import utils  # type: ignore

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    p = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(p.stderr or p.stdout).strip()}")

def next_version_tag(tag: str) -> str:
    # If no Product Finder image tag exists yet, the first build should become v1.
    match = re.fullmatch(r"v(\d+)", (tag or "").strip(), re.IGNORECASE)
    if not match:
        return "v1"
    return f"v{int(match.group(1)) + 1}"

def image_tag_env_key(agent_name: str) -> str:
    return f"PF_IMAGE_TAG_{agent_name.upper().replace('-', '_')}"

def get_latest_v_tag_from_acr(repository: str) -> str:
    # Query ACR as source of truth so notebook cells can run out of order safely.
    tags_out = run(
        f"az acr repository show-tags --name {ACR_NAME} --repository {repository} --output json",
        "",
        ""
    )
    if not tags_out.success:
        return ""

    tags = tags_out.json_data if isinstance(tags_out.json_data, list) else []
    latest = 0
    for tag in tags:
        if not isinstance(tag, str):
            continue
        match = re.fullmatch(r"v(\d+)", tag.strip(), re.IGNORECASE)
        if match:
            latest = max(latest, int(match.group(1)))

    return f"v{latest}" if latest > 0 else ""

# ── Load shared runtime config from environment ───────────────────────────────
SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
SPOKE_RG = azd_get("SPOKE_RESOURCE_GROUP")
SPOKE_ACCOUNT = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
SPOKE_PROJECT = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
ACR_NAME = azd_get("SPOKE_ACR_NAME")
ACR_SERVER = azd_get("SPOKE_ACR_LOGIN_SERVER")

PF_MODEL_CONNECTION = azd_get_optional("PF_MODEL_CONNECTION", "Product-Finder-DEV-LLM")
PF_MODEL_DEPLOYMENT = azd_get_optional("PF_MODEL_DEPLOYMENT", "gpt-5.4-mini")
PF_MODEL_FULL = azd_get_optional("PF_MODEL_FULL", f"{PF_MODEL_CONNECTION}/{PF_MODEL_DEPLOYMENT}")
PF_MODEL_APIM_PRODUCT_ID = azd_get_optional("PF_MODEL_APIM_PRODUCT_ID", "")
PF_MODEL_SUBSCRIPTION_NAME = azd_get_optional("PF_MODEL_SUBSCRIPTION_NAME", "")
PF_MODEL_SUBSCRIPTION_KEY = azd_get_optional("PF_MODEL_SUBSCRIPTION_KEY", "")
PF_MODEL_API_VERSION = azd_get_optional("PF_MODEL_API_VERSION", "2025-03-01-preview")
HUB_RG = azd_get_optional("PF_HUB_RG", "")
APIM_NAME = azd_get_optional("PF_APIM_NAME", "")
APIM_GATEWAY_URL = azd_get_optional("APIM_GATEWAY_URL", "").rstrip("/")
FOUNDRY_EP = azd_get_optional(
    "FOUNDRY_PROJECT_ENDPOINT",
    f"https://{SPOKE_ACCOUNT}.services.ai.azure.com/api/projects/{SPOKE_PROJECT}"
)

if not APIM_GATEWAY_URL and HUB_RG and APIM_NAME:
    gw_out = run(
        f"az apim show -g {HUB_RG} -n {APIM_NAME} --query gatewayUrl -o tsv",
        "APIM gateway resolved",
        "Failed to resolve APIM gateway URL"
    )
    if gw_out.success:
        APIM_GATEWAY_URL = (gw_out.stdout or "").strip().rstrip("/")

PF_MODEL_AZURE_ENDPOINT = APIM_GATEWAY_URL

PF_RAG_SEARCH_ENDPOINT = azd_get("PF_RAG_SEARCH_ENDPOINT")
PF_RAG_INDEX = azd_get("PF_RAG_INDEX")
PF_RAG_SEARCH_API_KEY = azd_get("PF_RAG_SEARCH_API_KEY")

FOUNDRY_ACCOUNT = SPOKE_ACCOUNT

AGENTS_BASE = pathlib.Path("../agents").resolve()
AGENTS_BASE.mkdir(exist_ok=True)

SPECIALIST_NAMES = [
    "pf-contextualizer",
    "pf-product-intelligence",
    "pf-compatibility",
    "pf-aligner",
    "pf-sample-request",
]

# Read current image tags from ACR as source of truth.
# Tag increment still happens only in the build cell when a new image is produced.
AGENT_IMAGE_TAG_KEYS = {name: image_tag_env_key(name) for name in SPECIALIST_NAMES}
AGENT_IMAGE_TAGS = {
    name: get_latest_v_tag_from_acr(name)
    for name in SPECIALIST_NAMES
}

if not PF_MODEL_CONNECTION or not PF_MODEL_FULL:
    raise RuntimeError("Missing PF_MODEL_CONNECTION / PF_MODEL_FULL. Run Notebook 1 first to create the model-access contract and Foundry connection.")
if not PF_MODEL_APIM_PRODUCT_ID or not PF_MODEL_SUBSCRIPTION_NAME:
    utils.print_warning("PF model-access contract metadata is missing from azd env. Re-run Notebook 1 to recreate the Citadel-style model-access contract outputs.")
if not PF_MODEL_SUBSCRIPTION_KEY:
    raise RuntimeError("Missing PF_MODEL_SUBSCRIPTION_KEY. Run Notebook 1 deploy cell to persist the APIM subscription key.")
if not PF_MODEL_AZURE_ENDPOINT:
    raise RuntimeError("Missing APIM gateway URL. Ensure APIM_GATEWAY_URL is present in azd env or PF_HUB_RG/PF_APIM_NAME are set.")

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = None

def ensure_project_client():
    global project_client
    if project_client is None:
        project_client = AIProjectClient(endpoint=FOUNDRY_EP, credential=DefaultAzureCredential(), allow_preview=True)
        utils.print_info("Initialized Foundry project client for this kernel session.")
    return project_client

utils.print_info(f"Model (via APIM): {PF_MODEL_FULL}")
utils.print_info(f"Foundry connection: {PF_MODEL_CONNECTION}")
utils.print_info(f"Model access contract product: {PF_MODEL_APIM_PRODUCT_ID or '<missing>'}")
utils.print_info(f"Model subscription: {PF_MODEL_SUBSCRIPTION_NAME or '<missing>'}")
utils.print_info(f"Model subscription key: {(PF_MODEL_SUBSCRIPTION_KEY[:8] + '...') if PF_MODEL_SUBSCRIPTION_KEY else '<missing>'}")
utils.print_info(f"Model API version: {PF_MODEL_API_VERSION}")
utils.print_info(f"Model endpoint: {PF_MODEL_AZURE_ENDPOINT}")
utils.print_info("Specialist image tags (current, live from ACR):")
for agent_name in SPECIALIST_NAMES:
    utils.print_info(f"  {agent_name}: {AGENT_IMAGE_TAGS[agent_name] or '<none>'}")
utils.print_info(f"ACR:                {ACR_SERVER}")
utils.print_info(f"Foundry endpoint:   {FOUNDRY_EP}")
utils.print_info(f"Search endpoint:    {PF_RAG_SEARCH_ENDPOINT}")
utils.print_info(f"Search index:       {PF_RAG_INDEX}")
utils.print_info(f"Agents base dir:    {AGENTS_BASE}")

## 0️⃣ Load environment and prerequisites

Resolve Azure subscription, Foundry project, ACR, APIM, and search configuration from azd environment.  
Validate that Notebook 1 and 2 have already run and persisted their outputs.
Prepare image tag tracking and specialist agent names.

### 1️⃣ Write specialist agent source files

Each agent's source code is written to `product-finder/agents/<agent-name>/`.  
You can inspect and edit these files before building the container images.

In [ ]:
REQUIREMENTS = """\
agent-framework>=1.2.0
agent-framework-openai>=1.8.1
agent-framework-foundry-hosting>=1.0.0a260507
azure-monitor-opentelemetry>=1.6.0
requests>=2.32.0
"""

DOCKERFILE = """\
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py .
EXPOSE 8088
CMD ["python", "main.py"]
"""

# Keep the checked-in agent source files as the source of truth for builds.
AGENT_NAMES = [
    "pf-contextualizer",
    "pf-product-intelligence",
    "pf-compatibility",
    "pf-aligner",
    "pf-sample-request",
]

AGENT_SOURCES = {}
for agent_name in AGENT_NAMES:
    source_path = AGENTS_BASE / agent_name / "main.py"
    if not source_path.exists():
        raise FileNotFoundError(f"Missing agent source file: {source_path}")
    AGENT_SOURCES[agent_name] = source_path.read_text(encoding="utf-8")

for agent_name, main_py in AGENT_SOURCES.items():
    d = AGENTS_BASE / agent_name
    d.mkdir(exist_ok=True)
    (d / "main.py").write_text(main_py.strip(), encoding="utf-8")
    (d / "requirements.txt").write_text(REQUIREMENTS, encoding="utf-8")
    (d / "Dockerfile").write_text(DOCKERFILE, encoding="utf-8")
    (d / "agent.yaml").write_text(
        f"kind: hosted\nname: {agent_name}\nprotocols:\n"
        f"  - protocol: responses\n    version: 1.0.0\n"
        f"resources:\n  cpu: \"1\"\n  memory: 2Gi\n"
        f"environment_variables:\n"
        f"  - name: PF_MODEL_DEPLOYMENT\n    value: ${{PF_MODEL_DEPLOYMENT}}\n"
        f"  - name: PF_MODEL_AZURE_ENDPOINT\n    value: ${{PF_MODEL_AZURE_ENDPOINT}}\n"
        f"  - name: PF_MODEL_SUBSCRIPTION_KEY\n    value: ${{PF_MODEL_SUBSCRIPTION_KEY}}\n"
        f"  - name: PF_MODEL_API_VERSION\n    value: ${{PF_MODEL_API_VERSION}}\n"
        f"  - name: PF_RAG_SEARCH_ENDPOINT\n    value: ${{PF_RAG_SEARCH_ENDPOINT}}\n"
        f"  - name: PF_RAG_INDEX\n    value: ${{PF_RAG_INDEX}}\n"
        f"  - name: PF_RAG_SEARCH_API_KEY\n    value: ${{PF_RAG_SEARCH_API_KEY}}\n"
        f"  - name: OTEL_SERVICE_NAME\n    value: {agent_name}\n"
        f"  - name: ENABLE_INSTRUMENTATION\n    value: \"true\"\n",
        encoding="utf-8"
    )
    utils.print_ok(f"  Source files written: {d.name}/")

### 2️⃣ Build all specialist images in Azure Container Registry

Increment image version tags in azd environment.  
Submit ACR builds for each specialist agent (runs server-side).  
Poll until all images appear in ACR (typically 2-3 min each).

In [ ]:
import time as _t

# ── 2️⃣  Build all specialist images in ACR ───────────────────────────────────
# Increment tags here, just before building, so deploy always uses the new image.
PREVIOUS_AGENT_IMAGE_TAGS = dict(AGENT_IMAGE_TAGS)
for agent_name in SPECIALIST_NAMES:
    env_key = AGENT_IMAGE_TAG_KEYS[agent_name]
    new_tag = next_version_tag(AGENT_IMAGE_TAGS[agent_name])
    set_azd_env(env_key, new_tag)
    AGENT_IMAGE_TAGS[agent_name] = new_tag
    utils.print_info(f"  {agent_name}: {PREVIOUS_AGENT_IMAGE_TAGS[agent_name]} -> {new_tag}")

# NOTE: --no-logs suppresses Windows encoding crash on streaming logs.
# The build runs server-side in ACR regardless.
for agent_name in SPECIALIST_NAMES:
    agent_tag = AGENT_IMAGE_TAGS[agent_name]
    utils.print_info(f"\nBuilding: {agent_name}:{agent_tag}")
    build_out = run(
        f"az acr build --registry {ACR_NAME} --resource-group {SPOKE_RG} "
        f"--image {agent_name}:{agent_tag} "
        f"--file {AGENTS_BASE / agent_name}/Dockerfile "
        f"--no-logs {AGENTS_BASE / agent_name}/",
        f"{agent_name} build queued", f"{agent_name} build failed"
    )
    if not build_out.success:
        # Roll back the tag on failure
        set_azd_env(AGENT_IMAGE_TAG_KEYS[agent_name], PREVIOUS_AGENT_IMAGE_TAGS[agent_name])
        raise RuntimeError(f"ACR build failed for {agent_name}")

# Poll until all images appear in ACR (build takes ~2-3 min each)
utils.print_info("\nPolling ACR for image presence (builds run server-side)...")
for agent_name in SPECIALIST_NAMES:
    agent_tag = AGENT_IMAGE_TAGS[agent_name]
    for attempt in range(30):   # up to 5 min
        tag_out = run(
            f"az acr repository show-tags --name {ACR_NAME} "
            f"--repository {agent_name} --output json", "", ""
        )
        if tag_out.success and tag_out.json_data and agent_tag in tag_out.json_data:
            utils.print_ok(f"  {agent_name}:{agent_tag}  verified in ACR")
            break
        utils.print_info(f"  {agent_name}: waiting... ({(attempt+1)*10}s)")
        _t.sleep(10)
    else:
        raise RuntimeError(f"Image {agent_name}:{agent_tag} not found in ACR after 5 min")

### 3️⃣ Deploy specialist agents to Foundry and wait for active status

In [ ]:
import time as _t
from azure.ai.projects.models import HostedAgentDefinition

project_client  = ensure_project_client()
utils.print_ok(f"Foundry project client ready: {FOUNDRY_EP}")

deployed_agents = {}
for agent_name in SPECIALIST_NAMES:
    agent_tag = AGENT_IMAGE_TAGS[agent_name]
    image = f"{ACR_SERVER}/{agent_name}:{agent_tag}"
    utils.print_info(f"\nDeploying {agent_name}  →  {image}")
    agent_version = project_client.agents.create_version(
        agent_name=agent_name,
        definition=HostedAgentDefinition({
            "container_protocol_versions": [{"protocol": "responses", "version": "1.0.0"}],
            "image": image,
            "cpu": "1",
            "memory": "2Gi",
            "environment_variables": {
                "PF_MODEL_DEPLOYMENT": PF_MODEL_DEPLOYMENT,
                "PF_MODEL_AZURE_ENDPOINT": PF_MODEL_AZURE_ENDPOINT,
                "PF_MODEL_SUBSCRIPTION_KEY": PF_MODEL_SUBSCRIPTION_KEY,
                "PF_MODEL_API_VERSION": PF_MODEL_API_VERSION,
                "PF_RAG_SEARCH_ENDPOINT": PF_RAG_SEARCH_ENDPOINT,
                "PF_RAG_INDEX": PF_RAG_INDEX,
                "PF_RAG_SEARCH_API_KEY": PF_RAG_SEARCH_API_KEY,
                "OTEL_SERVICE_NAME": agent_name,
                "ENABLE_INSTRUMENTATION": "true",
                "ENABLE_SENSITIVE_DATA": "true",
            },
        }),
    )
    deployed_agents[agent_name] = agent_version
    utils.print_ok(f"  Created version {agent_version.version}  (status: {agent_version.status})")

# ── Poll until all active ─────────────────────────────────────────────────────
utils.print_info("\nWaiting for all agents to become active (2-5 min each)...")
MAX_WAIT = 600
POLL     = 15
active_agents = {}

for agent_name, av in deployed_agents.items():
    elapsed = 0
    while elapsed < MAX_WAIT:
        v = project_client.agents.get_version(agent_name=av.name, agent_version=av.version)
        utils.print_info(f"  [{elapsed:>3}s] {agent_name}: {v.status}")
        if v.status == "active":
            active_agents[agent_name] = v
            utils.print_ok(f"  {agent_name} is ACTIVE")
            break
        if v.status in ("failed", "error"):
            raise RuntimeError(f"{agent_name} deployment failed: {v.status}")
        _t.sleep(POLL); elapsed += POLL
    else:
        raise RuntimeError(f"Timed out waiting for {agent_name} to become active")

### 4️⃣ Assign Foundry User RBAC to each agent identity

Extract the managed identity (service principal) for each deployed agent.  
Assign `Foundry User` role on the Foundry account scope so agents can call other Foundry services.  
Wait 60s for RBAC propagation if new assignments were created.

In [ ]:
# ── 4️⃣  Assign Foundry User RBAC to each agent identity ─────────────────────
foundry_scope = (
    f"/subscriptions/{SUB_ID}/resourceGroups/{SPOKE_RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_ACCOUNT}"
)
new_assignments_created = 0
for agent_name, v in active_agents.items():
    identity = v.instance_identity
    if not identity:
        agent_detail = project_client.agents.get(agent_name=agent_name)
        identity = agent_detail.instance_identity
    if not identity:
        utils.print_warning(f"  {agent_name}: could not resolve identity - assign Foundry User manually")
        continue

    pid = identity.principal_id
    check = subprocess.run(
        ["az", "role", "assignment", "list", "--assignee-object-id", pid,
         "--role", "Foundry User", "--scope", foundry_scope, "--query", "[0].id", "-o", "tsv"],
        capture_output=True, text=True, shell=True
    )
    if check.stdout.strip():
        utils.print_ok(f"  {agent_name}: Foundry User role already assigned")
    else:
        result = subprocess.run(
            ["az", "role", "assignment", "create",
             "--assignee-object-id", pid,
             "--assignee-principal-type", "ServicePrincipal",
             "--role", "Foundry User",
             "--scope", foundry_scope, "--only-show-errors", "-o", "none"],
            capture_output=True, text=True, shell=True
        )
        if result.returncode == 0:
            new_assignments_created += 1
            utils.print_ok(f"  {agent_name}: Foundry User assigned (principal: {pid[:8]}...)")
        else:
            utils.print_warning(f"  {agent_name}: RBAC assignment failed - {result.stderr.strip()}")

if new_assignments_created > 0:
    utils.print_info("\nWaiting 60s for RBAC propagation...")
    _t.sleep(60)
    utils.print_ok("RBAC propagation wait complete.")
else:
    utils.print_info("\nNo new RBAC assignments created; skipping propagation wait.")

### 5️⃣ Smoke test each specialist agent directly

Invoke each specialist with a representative test input via the OpenAI client.  
Implement exponential retry with jitter for transient failures (queued, timeout, 5xx errors).  
Persist the list of deployed specialist names to azd environment for downstream notebooks.  
Raise an error if any smoke test fails definitively.

In [ ]:
# ── 5️⃣  Smoke test each specialist agent directly ───────────────────────────
import uuid
import time as _t
import random

if "ensure_project_client" not in globals():
    raise RuntimeError("Cell 2 must be run first (missing ensure_project_client).")
project_client = ensure_project_client()

print("=" * 65)
print("SPECIALIST AGENT SMOKE TESTS")
print("=" * 65)

smoke_tests = {
    "pf-contextualizer":
        "I need a product to wash my dog",
    "pf-product-intelligence":
        '{"intent":"recommendation","entities":{"animal_type":"dog","age_group":"adult","condition":"healthy"},"contextualized_query":"shampoo for a healthy adult dog"}',
    "pf-compatibility":
        "Can I use SynPet Clean Pro together with SynPet Odor Shield?",
    "pf-aligner":
        '{"draft":"SynPet Clean Pro is recommended","original_intent":"recommendation for dog shampoo","persona":"external_customer"}',
    "pf-sample-request":
        '[GOVERNANCE CONTEXT]\npersona: external_customer\n\nI would like a sample of SynPet Gentle Care.',
}

MAX_SMOKE_RETRIES = 7
BASE_RETRY_WAIT_SECONDS = 6
MAX_RETRY_WAIT_SECONDS = 45
smoke_failures = {}

TRANSIENT_HINTS = (
    "session_not_ready",
    "server_error",
    "internal server error",
    "temporarily unavailable",
    "timeout",
    "timed out",
    "connection reset",
    "connection aborted",
    "bad gateway",
    "gateway timeout",
    "rate limit",
    "too many requests",
    "429",
)

TRANSIENT_STATUSES = {
    "queued",
    "in_progress",
    "failed",
    "error",
    "incomplete",
}


def is_transient_failure(resp_status: str, err_text: str) -> bool:
    if resp_status and resp_status in TRANSIENT_STATUSES:
        return True
    low = (err_text or "").lower()
    return any(hint in low for hint in TRANSIENT_HINTS)


for agent_name, test_input in smoke_tests.items():
    last_error = None
    for attempt in range(1, MAX_SMOKE_RETRIES + 1):
        try:
            # Re-acquire the OpenAI client per attempt to avoid stale transport state.
            oclient = project_client.get_openai_client(agent_name=agent_name)
            resp = oclient.responses.create(
                input=test_input,
                metadata={"conversation_id": str(uuid.uuid4())},
            )
            resp_status = (getattr(resp, "status", "") or "").lower()
            output = (resp.output_text or "").strip()

            if resp_status and resp_status != "completed":
                raise RuntimeError(f"response status={resp_status}; error={getattr(resp, 'error', None)}")
            if not output:
                raise RuntimeError(f"empty output; response status={resp_status or '<unknown>'}; error={getattr(resp, 'error', None)}")

            preview = output.replace("\n", " ")[:200]
            utils.print_ok(f"  {agent_name}: {preview}...")
            last_error = None
            break
        except Exception as e:
            last_error = e
            err_text = str(e)
            resp_status = ""
            if "response status=" in err_text.lower():
                try:
                    resp_status = err_text.split("response status=", 1)[1].split(";", 1)[0].strip().lower()
                except Exception:
                    resp_status = ""

            transient = is_transient_failure(resp_status=resp_status, err_text=err_text)
            if transient and attempt < MAX_SMOKE_RETRIES:
                wait = min(MAX_RETRY_WAIT_SECONDS, BASE_RETRY_WAIT_SECONDS * (2 ** (attempt - 1)))
                wait = wait + random.uniform(0, 2)
                utils.print_warning(
                    f"  {agent_name}: transient smoke-test failure "
                    f"(attempt {attempt}/{MAX_SMOKE_RETRIES}) -> {err_text}; retrying in {wait:.1f}s"
                )
                _t.sleep(wait)
                continue
            break

    if last_error is not None:
        smoke_failures[agent_name] = str(last_error)
        utils.print_warning(f"  {agent_name}: smoke test failed — {last_error}")

# Save deployed specialist names for downstream notebooks.
# If Cell 7 was not run in this kernel, fall back to the smoke-test set.
if "active_agents" in globals() and isinstance(active_agents, dict) and active_agents:
    deployed_names = list(active_agents.keys())
else:
    deployed_names = list(smoke_tests.keys())
    utils.print_warning("active_agents not found in kernel; using smoke-test specialist list for PF_DEPLOYED_SPECIALISTS.")

deployed_json = json.dumps(deployed_names)
save_deployed = subprocess.run(["azd", "env", "set", "PF_DEPLOYED_SPECIALISTS", deployed_json], capture_output=True, text=True)
if save_deployed.returncode != 0:
    raise RuntimeError(f"Failed to persist PF_DEPLOYED_SPECIALISTS: {(save_deployed.stderr or save_deployed.stdout).strip()}")
utils.print_ok("Persisted PF_DEPLOYED_SPECIALISTS to azd env")

if smoke_failures:
    failed = ", ".join(smoke_failures.keys())
    raise RuntimeError(f"Smoke tests failed for: {failed}. Check logs for details.")

print()
utils.print_ok("✅ All specialists deployed and smoke tested. Proceed to Notebook 4.")